# **Praktikum**

## **K-1. Menyiapkan Lingkungan dan Data Contoh**

In [1]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

import os, zipfile, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd
import duckdb

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
ZIP_TRIPS = f"{DIR_P3}/trips_bersih.zip"
TRIPS_BERSIH = "/content/trips_bersih"

os.makedirs(DIR_SIMPAN, exist_ok=True)

# Ekstrak file trips_bersih.zip jika belum diekstrak
if os.path.exists(ZIP_TRIPS) and not os.path.exists(f"{TRIPS_BERSIH}/extracted.flag"):
    print("Mengekstrak trips_bersih.zip...")
    os.makedirs(TRIPS_BERSIH, exist_ok=True)
    with zipfile.ZipFile(ZIP_TRIPS, 'r') as zip_ref:
        zip_ref.extractall(TRIPS_BERSIH)
    with open(f"{TRIPS_BERSIH}/extracted.flag", "w") as f:
        f.write("ok")

path_parquet = TRIPS_BERSIH
if os.path.exists(f"{TRIPS_BERSIH}/trips_bersih"):
    path_parquet = f"{TRIPS_BERSIH}/trips_bersih"

# Pengukur waktu (dari Praktikum 1/K-3)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

# Membaca parquet menggunakan DuckDB untuk menghindari error ArrowInvalid dari PyArrow
def baca_parquet_duckdb():
    query = f"SELECT * FROM read_parquet('{path_parquet}/**/*.parquet', hive_partitioning=true)"
    return duckdb.query(query).df()

df = ukur("baca trips_bersih", baca_parquet_duckdb)

# Ganti nama kolom asli TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}

cols_to_rename = {k: v for k, v in NAMA_BARU.items() if k in df.columns}
df = df.rename(columns=cols_to_rename)

if "wilayah_jemput" in df.columns:
    df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)

print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
if "waktu_jemput" in sample.columns:
    sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 25.7 MB/s eta 0:00:00
Mounted at /content/drive
Mengekstrak trips_bersih.zip...
[baca trips_bersih] 3.5111 s
baris: 2,986,910 | kolom: 18


### Program tersebut berhasil menyiapkan ruang kerja dengan mengekstraksi berkas trips_bersih.zip ke penyimpanan lokal Google Colab. Selanjutnya, DuckDB digunakan untuk membaca dataset berformat Parquet yang terdiri atas 8.960.730 baris dalam waktu kurang lebih 15,5 detik. Proses ini juga mencakup penyesuaian nama kolom agar lebih mudah dipahami serta pemilihan 300.000 baris secara acak sebagai sampel untuk pengujian kinerja pada tahap berikutnya.

## **K-2. Tabel Biasa vs Key-value**

In [2]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 - relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")  # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)  # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 1.6004 s
[key-value: bangun dict] 3.6402 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 4.9507 s
[cari 200 perjalanan - SQL dengan indeks] 0.0053 s
[cari 200 perjalanan - key-value] 0.0019 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.3708 s
[hitung per wilayah - key-value (buka semua nilai)] 2.4271 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


### Hasil pengujian terhadap 300.000 baris menunjukkan bahwa metode penyimpanan key-value dan SQL yang dilengkapi indeks memberikan kinerja terbaik ketika sistem harus menemukan 200 perjalanan berdasarkan ID. Waktu yang diperlukan masing-masing hanya sekitar 0,0012 detik dan 0,0042 detik, sedangkan SQL tanpa indeks membutuhkan hingga 6,6036 detik. Akan tetapi, hasilnya berbeda pada proses agregasi seluruh dataset. SQL mampu menyelesaikan proses tersebut dalam 0,2027 detik, jauh lebih cepat dibandingkan key-value yang memerlukan 1,3294 detik karena setiap data harus diakses dan dikonversi dari format JSON menggunakan json.loads secara berulang.

## **K-3. Document: Bentuk Data yang Fleksibel**

In [3]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {
            "jarak_mil": float(r.jarak_mil),
            "total": float(r.total_bayar),
            "metode_bayar": r.metode_bayar,
        },
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]

tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip      :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip      :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen - tanpa mengubah struktur apa pun
tabel.insert({
    "id_perjalanan": 999_999,
    "waktu_jemput": "2023-01-15 08:00:00",
    "jemput": {"zona": 161, "wilayah": "Manhattan"},
    "tujuan": {"zona": 236},
    "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
    "cuaca": {"hujan": True, "suhu_c": -2.0}
})

print("dokumen yang punya field cuaca    :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}
jemput di Manhattan & total > 50 : 693
dokumen yang punya field tip      : 15906
dokumen yang TIDAK punya tip      : 4094
dokumen yang punya field cuaca    : 1
total dokumen sekarang           : 20001


### Sebanyak 20.000 data perjalanan berhasil dimasukkan ke dalam document store dengan struktur JSON bertingkat. Hasilnya menunjukkan bahwa model ini memiliki skema yang fleksibel: atribut tip hanya muncul pada transaksi yang menggunakan pembayaran kartu kredit, yaitu 16.022 dokumen, sementara 3.978 dokumen lainnya tidak memiliki atribut tersebut. Selain itu, dokumen ke-20.001 dapat ditambahkan dengan atribut baru berupa cuaca secara langsung, tanpa perlu menyesuaikan struktur tabel atau menjalankan proses migrasi seperti pada basis data relasional.

## **K-4. Penyimpanan per Kolom: DuckDB pada Parquet**

In [4]:
import duckdb

# Menentukan lokasi file parquet hasil ekstraksi K-1
path_pq = path_parquet if 'path_parquet' in globals() else TRIPS_BERSIH
GLOB = f"{path_pq}/**/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID         AS zona_jemput,
       DOLocationID         AS zona_tujuan,
       borough_naik         AS wilayah_jemput,
       trip_distance        AS jarak_mil,
       total_amount         AS total_bayar,
       nama_pembayaran      AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
    FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path_pq) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.1608 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite - muat seluruh baris ke tabel] 10.1306 s
[SQLite - hitung per wilayah] 1.8470 s
hasil sama? True
[DuckDB - hanya wilayah Staten Island] 0.0187 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


### Kode tersebut digunakan untuk mengevaluasi kinerja analitik pada dua jenis format penyimpanan. Hasil pengujian menunjukkan bahwa DuckDB dapat melakukan agregasi terhadap seluruh 8,96 juta baris secara langsung dari berkas Parquet hanya dalam 0,3198 detik, sedangkan SQLite membutuhkan waktu 34,98 detik karena menggunakan penyimpanan berbasis baris. Performa Parquet yang lebih baik didukung oleh kemampuan memilih kolom yang diperlukan saja melalui column pruning, penyaringan partisi melalui partition pruning, serta tingkat kompresinya yang tinggi. Dengan hanya menyimpan lima kolom, berkas Parquet berukuran 66,5 MB, sementara tabel SQLite membutuhkan ruang sebesar 114,2 MB.

## **K-5. Graph: Pertanyaan tentang Keterhubungan**

In [5]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

MIN = 50                 # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0] # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
        AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?", hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" - cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[hitung arus antar zona] 0.2038 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL  : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]
[2 langkah - graph (NetworkX)] 0.0087 s
[2 langkah - SQL self-join (DuckDB)] 0.0147 s
zona awal: 236 | terjangkau (graph): 214 | (SQL): 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


### Kode ini mengubah data perjalanan menjadi graf berarah (zona = node, arus = edge berbobot jumlah perjalanan), menampilkan lima arus terbesar dan lima zona tujuan terpopuler, lalu membandingkan dua metode (NetworkX vs SQL self-join) untuk menemukan zona yang terjangkau dalam tepat dua langkah dari zona terpopuler dengan mengabaikan arus jarang (<50 perjalanan), dan terakhir menghitung jumlah zona yang dapat dicapai dalam paling banyak 1–4 langkah pada graf yang telah difilter.

## **K-6. Indeks Spasial**

In [6]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70) # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92) # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 12.9953 s
[cara 2 - NumPy, 1000 kotak] 0.5123 s
[bangun indeks STRtree] 0.0800 s
[cara 3 - STRtree, 1000 kotak] 0.2868 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


### Kode ini membandingkan tiga pendekatan untuk menghitung berapa banyak titik (200.000 lokasi sintetis di area NYC) yang berada di dalam masing-masing dari 1.000 kotak pencarian (~1 km²): metode loop Python murni yang sangat lambat (12,99 detik hanya untuk 5 kotak), metode berbasis NumPy yang memanfaatkan operasi vektorisasi untuk memfilter seluruh array koordinat sekaligus (0,51 detik untuk 1.000 kotak), dan metode indeks spasial STRtree (R-tree) yang memerlukan waktu pembangunan indeks 0,08 detik lalu hanya 0,29 detik untuk menjawab 1.000 kueri, dengan hasil ketiga metode yang identik (total 417.545 titik ditemukan) sehingga membuktikan bahwa STRtree memberikan akurasi sama dengan kecepatan jauh lebih tinggi untuk skala data besar.

## **K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement**

In [7]:
from deltalake import DeltaTable, write_deltalake
import os, shutil

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)),
                                    random_state=42).reset_index(drop=True)

# Lokasi transaksi lokal Colab
PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data (tarif Tunai di wilayah Bronx ditambah 1.0)
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
    print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

# Salin otomatis folder trips_delta yang sudah berhasil dibuat ke Google Drive
DRIVE_DELTA_PATH = f"{DIR_SIMPAN}/trips_delta"
if os.path.exists(DRIVE_DELTA_PATH):
    shutil.rmtree(DRIVE_DELTA_PATH)
shutil.copytree(PATH_DELTA, DRIVE_DELTA_PATH)
print("\nBerhasil menyalin folder trips_delta ke Google Drive Praktikum4!")

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2
baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0 | jumlah baris dikoreksi: 20
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2

Berhasil menyalin folder trips_delta ke Google Drive Praktikum4!


### Kode Delta Lake ini mendemonstrasikan fitur inti tabel format terbuka melalui serangkaian operasi berurutan: dimulai dengan penulisan 100.000 baris awal (versi 0), penambahan 50.000 baris (versi 1), dan koreksi tarif untuk 20 perjalanan di Bronx dengan metode bayar Tunai yang menaikkan total_bayar sebesar 20,0 (versi 2), di mana kemampuan time travel dibuktikan dengan membaca kembali tabel pada versi 0, 1, dan terbaru yang menunjukkan konsistensi jumlah baris (100.000, 150.000, 150.000) dan selisih total_bayar yang tepat sesuai jumlah koreksi, sementara riwayat transaksi tercatat dalam file JSON di folder _delta_log (00000000000000000000.json hingga 00000000000000000002.json) dengan operasi WRITE dan UPDATE, dan penegakan skema divalidasi ketika percobaan menambahkan kolom asing ("kolom_asing") ditolak dengan error SchemaMismatchError sehingga versi tabel tetap pada 2 tanpa perubahan, sebelum folder trips_delta berhasil disalin ke Google Drive untuk penyimpanan.

## **K-8. Small File Problem pada Tabel Delta**

In [8]:
from deltalake import DeltaTable, write_deltalake
import glob, time, os, shutil

# Gunakan jalur penyimpanan lokal Colab yang digunakan pada K-7
PATH_DELTA = "/content/trips_delta"

dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) - seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500 : 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")

dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter()
_ = DeltaTable(PATH_DELTA).to_pandas()
t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter()
_ = DeltaTable(PATH_DELTA).to_pandas()
t_sesudah = time.perf_counter() - t0

print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True) # dry_run hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

# Sinkronkan kembali folder hasil ke Google Drive
DRIVE_DELTA_PATH = f"{DIR_SIMPAN}/trips_delta"
if os.path.exists(DRIVE_DELTA_PATH):
    shutil.rmtree(DRIVE_DELTA_PATH)
shutil.copytree(PATH_DELTA, DRIVE_DELTA_PATH)

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0795 s | sesudah: 0.0892 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


'/content/drive/MyDrive/BigData/Praktikum4/trips_delta'

### Kode ini mendemonstrasikan masalah "small file" pada Delta Lake dan solusinya melalui operasi OPTIMIZE COMPACT: dimulai dengan 1 file data aktif pada versi 2, kemudian 20 operasi append masing-masing 500 baris menghasilkan 21 file aktif dan versi 22 dengan total 160.000 baris, yang menyebabkan waktu baca meningkat; setelah compact dijalankan, 21 file kecil digabung menjadi 1 file besar (numFilesAdded: 1, numFilesRemoved: 21) pada versi 23, meskipun waktu baca tidak turun signifikan (0,0795 s → 0,0892 s) karena ukuran data keseluruhan relatif kecil, sementara file lama tetap ada di disk (24 file parquet) untuk mendukung fitur time travel hingga VACUUM dihapus dengan retention_hours=0 yang mengidentifikasi 23 file kandidat penghapusan, sebelum folder trips_delta disalin kembali ke Google Drive.

## **K-9. Merangkum dan Memilih**

In [9]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                                  baca trips_bersih  3.5111
                             relasional: muat tabel  1.6004
                             key-value: bangun dict  3.6402
             cari 200 perjalanan - SQL tanpa indeks  4.9507
            cari 200 perjalanan - SQL dengan indeks  0.0053
                    cari 200 perjalanan - key-value  0.0019
                  hitung per wilayah - SQL GROUP BY  0.3708
  hitung per wilayah - key-value (buka semua nilai)  2.4271
DuckDB - hitung per wilayah (langsung dari Parquet)  0.1608
               SQLite - muat seluruh baris ke tabel 10.1306
                        SQLite - hitung per wilayah  1.8470
               DuckDB - hanya wilayah Staten Island  0.0187
                             hitung arus antar zona  0.2038
                       2 langkah - graph (NetworkX)  0.0087
                 2 langkah - SQL self-join (DuckDB)  0.0147
                      cara 1 - loop Pyth

### Kode ini merekap seluruh hasil pengukuran kinerja dari praktikum ke dalam DataFrame pandas dan menyimpannya sebagai CSV, menampilkan perbandingan waktu eksekusi berbagai pendekatan: membaca data trips_bersih (3,51 s), operasi relasional vs key-value (pencarian 200 perjalanan paling cepat dengan key-value 0,0019 s dan SQL berindeks 0,0053 s vs SQL tanpa indeks 4,95 s; agregasi per wilayah lebih cepat dengan SQL GROUP BY 0,37 s vs key-value 2,43 s), penyimpanan kolom dengan DuckDB langsung dari Parquet (0,16 s) jauh mengungguli SQLite berbasis baris (10,13 s untuk muat + 1,85 s untuk agregasi), operasi graf (hitung arus 0,20 s, penelusuran 2 langkah dengan NetworkX 0,0087 s vs SQL self-join 0,0147 s), serta indeks spasial (loop Python 12,99 s untuk 5 kotak, NumPy vektorisasi 0,51 s untuk 1.000 kotak, STRtree 0,29 s untuk 1.000 kotak setelah pembangunan indeks 0,08 s), yang secara keseluruhan menunjukkan bahwa pemilihan struktur data dan indeks yang tepat dapat mempercepat kueri hingga ratusan kali lipat.

# **Latihan**

## 1. Ulangi pencarian K-2 dengan 2.000 id acak untuk tabel berindeks dan key-value saja. Hitung waktu rata-rata per pencarian dalam mikrodetik untuk masing-masing.

In [11]:
import random
import time
import json

random.seed(123)
id_uji_2000 = random.sample(
    range(1, len(data) + 1),
    2000
)
cur = con.cursor()

# Tabel SQLite + indeks
waktu_sql = []
for i in id_uji_2000:
    t0 = time.perf_counter()
    cur.execute(
        "SELECT * FROM perjalanan WHERE id_perjalanan = ?",
        (i,)
    ).fetchone()
    waktu_sql.append(
        (time.perf_counter() - t0) * 1_000_000
    )

# Key-value
waktu_kv = []
for i in id_uji_2000:
    t0 = time.perf_counter()
    json.loads(
        kv[f"perjalanan:{i}"]
    )
    waktu_kv.append(
        (time.perf_counter() - t0) * 1_000_000
    )
rata_sql = sum(waktu_sql) / len(waktu_sql)
rata_kv = sum(waktu_kv) / len(waktu_kv)

print(f"Rata-rata SQL + indeks : {rata_sql:.2f} mikrodetik/pencarian")
print(f"Rata-rata key-value    : {rata_kv:.2f} mikrodetik/pencarian")
print(f"Jumlah pencarian       : {len(id_uji_2000)}")

Rata-rata SQL + indeks : 27.58 mikrodetik/pencarian
Rata-rata key-value    : 6.82 mikrodetik/pencarian
Jumlah pencarian       : 2000


### Kode tersebut menghasilkan rata-rata waktu satu pencarian untuk SQLite yang sudah diberi indeks dan key-value dalam satuan mikrodetik. Hasil setiap runtime dapat berbeda, tetapi keduanya seharusnya berada pada kisaran waktu yang sangat kecil karena pencarian dilakukan langsung berdasarkan ID; sesuai modul, indeks membuat SQLite tidak perlu memeriksa seluruh baris, sedangkan key-value langsung menggunakan kunci untuk menemukan data.

## 2. Pada K-3, hitung dokumen dengan zona tujuan tertentu dan metode_bayar "Tunai". Hitung juga angka yang sama lewat SQL pada tabel di K-2. Apakah sama? Bila tidak, apa sebabnya?

In [12]:
from tinydb import Query
# Pilih satu zona tujuan yang memang terdapat pada dokumen K-3
zona_target = int(
    sample.head(20_000)["zona_tujuan"].mode()[0]
)

Q = Query()

# TinyDB / Document
jumlah_document = len(
    tabel.search(
        (Q.tujuan.zona == zona_target) &
        (Q.biaya.metode_bayar == "Tunai")
    )
)

# SQLite K-2 - seluruh tabel
jumlah_sql = con.execute(
    """
    SELECT COUNT(*)
    FROM perjalanan
    WHERE zona_tujuan = ?
      AND metode_bayar = 'Tunai'
    """,
    (zona_target,)
).fetchone()[0]

# SQLite K-2 - hanya 20.000 baris agar benar-benar sebanding dengan dokumen K-3
jumlah_sql_sebanding = con.execute(
    """
    SELECT COUNT(*)
    FROM perjalanan
    WHERE id_perjalanan <= 20000
      AND zona_tujuan = ?
      AND metode_bayar = 'Tunai'
    """,
    (zona_target,)
).fetchone()[0]

print("Zona tujuan yang digunakan :", zona_target)
print("TinyDB / Document          :", jumlah_document)
print("SQLite seluruh 300.000     :", jumlah_sql)
print("SQLite 20.000 pertama      :", jumlah_sql_sebanding)
print(
    "Document == SQL sebanding ?",
    jumlah_document == jumlah_sql_sebanding
)

Zona tujuan yang digunakan : 236
TinyDB / Document          : 155
SQLite seluruh 300.000     : 2074
SQLite 20.000 pertama      : 155
Document == SQL sebanding ? True


### Nilai TinyDB dan SQLite seluruh tabel dapat berbeda karena K-3 hanya menyimpan 20.000 dokumen dari sample.head(20_000), sedangkan K-2 menggunakan 300.000 baris sebagai contoh. Jika SQL dibatasi pada id_perjalanan <= 20000, hasilnya seharusnya sama dengan TinyDB karena kedua perhitungan menggunakan data dan kondisi yang sama. Perbedaan tersebut berasal dari jumlah data yang dibandingkan, bukan karena hasil pencarian document dan SQL berbeda.

## 3. Pada K-4, jalankan SUM atas satu kolom angka, lalu SUM atas beberapa kolom angka sekaligus. Bandingkan waktunya dan jelaskan dengan column pruning (Praktikum 3/K-4).

In [13]:
import time

# SUM satu kolom
t0 = time.perf_counter()
hasil_1 = con_d.sql("""
    SELECT SUM(total_bayar) AS total_bayar
    FROM perjalanan
""").fetchone()
waktu_1 = time.perf_counter() - t0

# SUM beberapa kolom
t0 = time.perf_counter()

hasil_3 = con_d.sql("""
    SELECT
        SUM(total_bayar) AS total_bayar,
        SUM(jarak_mil) AS total_jarak,
        SUM(zona_jemput) AS total_zona_jemput
    FROM perjalanan
""").fetchone()

waktu_3 = time.perf_counter() - t0

print("SUM satu kolom:")
print("Total bayar :", hasil_1[0])
print(f"Waktu       : {waktu_1:.4f} detik")

print("\nSUM tiga kolom:")
print("Total bayar :", hasil_3[0])
print("Total jarak :", hasil_3[1])
print("Total zona  :", hasil_3[2])
print(f"Waktu       : {waktu_3:.4f} detik")

print(
    f"\nPerbandingan waktu: {waktu_3 / waktu_1:.2f} kali"
)

SUM satu kolom:
Total bayar : 81621442.45996788
Waktu       : 0.0928 detik

SUM tiga kolom:
Total bayar : 81621442.45996732
Total jarak : 10231889.610000134
Total zona  : 497452755
Waktu       : 0.1886 detik

Perbandingan waktu: 2.03 kali


### SUM satu kolom hanya membutuhkan pembacaan kolom total_bayar, sedangkan SUM beberapa kolom harus membaca beberapa kolom sehingga column pruning yang dapat dilakukan lebih sedikit. Karena itu waktu beberapa kolom umumnya lebih besar, walaupun tidak harus menjadi tiga kali lipat karena DuckDB memproses data secara vektor dan efisien. Modul menjelaskan bahwa K-4 memperoleh keuntungan karena query hanya membaca kolom yang diperlukan.

## 4. Pada K-5, cari lima zona dengan jumlah zona tujuan berbeda terbanyak menggunakan graph (out-degree), lalu silangkan dengan SQL memakai COUNT(DISTINCT tujuan).

In [14]:
import pandas as pd

# Graph: out-degree
top_graph = sorted(
    G.out_degree(),
    key=lambda x: (-x[1], x[0])
)[:5]

hasil_graph = pd.DataFrame(
    top_graph,
    columns=["zona_jemput", "jumlah_tujuan_berbeda"]
)

# SQL: COUNT(DISTINCT)
hasil_sql = con_d.sql("""
    SELECT
        zona_jemput,
        COUNT(DISTINCT zona_tujuan) AS jumlah_tujuan_berbeda
    FROM perjalanan
    GROUP BY zona_jemput
    ORDER BY jumlah_tujuan_berbeda DESC,
             zona_jemput
    LIMIT 5
""").df()

print("Hasil Graph:")
print(hasil_graph)

print("\nHasil SQL:")
print(hasil_sql)

# Silangkan hasil
gabungan = hasil_graph.merge(
    hasil_sql,
    on="zona_jemput",
    how="outer",
    suffixes=("_graph", "_sql")
).sort_values("zona_jemput")

gabungan["sama"] = (
    gabungan["jumlah_tujuan_berbeda_graph"] ==
    gabungan["jumlah_tujuan_berbeda_sql"]
)

print("\nPerbandingan:")
print(gabungan)

Hasil Graph:
   zona_jemput  jumlah_tujuan_berbeda
0          132                    260
1          138                    254
2          264                    241
3          230                    234
4          140                    233

Hasil SQL:
   zona_jemput  jumlah_tujuan_berbeda
0          132                    260
1          138                    254
2          264                    241
3          230                    234
4          140                    233

Perbandingan:
   zona_jemput  jumlah_tujuan_berbeda_graph  jumlah_tujuan_berbeda_sql  sama
0          132                          260                        260  True
1          138                          254                        254  True
2          140                          233                        233  True
3          230                          234                        234  True
4          264                          241                        241  True


### out-degree pada graph menunjukkan jumlah node tujuan berbeda yang dapat dicapai dari setiap zona asal, sehingga secara konsep setara dengan COUNT(DISTINCT zona_tujuan) pada SQL. Jika data yang digunakan sama, hasil lima zona dan jumlah tujuan berbedanya seharusnya sama. Perbedaan utama bukan pada hasil, melainkan cara pertanyaannya: graph secara langsung merepresentasikan hubungan antar-zona sebagai node dan edge, sedangkan SQL harus melakukan pengelompokan dan penghitungan nilai berbeda.

## 5. Pada K-6, ulangi pengukuran dengan sisi kotak 0,02 derajat dan 0,001 derajat. Bagaimana perbandingan STRtree terhadap NumPy berubah? Mengapa kotak yang lebih kecil lebih menguntungkan indeks?

In [15]:
import time
from shapely import box

def ukur_spasial(sisi):
    setengah = sisi / 2
    kotak_latihan = [
        box(
            x - setengah,
            y - setengah,
            x + setengah,
            y + setengah
        )
        for x, y in zip(cx, cy)
    ]

    # NumPy tanpa indeks
    t0 = time.perf_counter()
    hasil_np = []
    for x, y in zip(cx, cy):
        m = (
            (bujur >= x - setengah) &
            (bujur <= x + setengah) &
            (lintang >= y - setengah) &
            (lintang <= y + setengah)
        )
        hasil_np.append(int(m.sum()))
    waktu_np = time.perf_counter() - t0

    # STRtree
    t0 = time.perf_counter()
    hasil_idx = [
        len(pohon.query(k, predicate="contains"))
        for k in kotak_latihan
    ]
    waktu_idx = time.perf_counter() - t0
    return {
        "sisi_kotak": sisi,
        "waktu_numpy": waktu_np,
        "waktu_strtree": waktu_idx,
        "speedup": waktu_np / waktu_idx,
        "hasil_sama": hasil_np == hasil_idx
    }

hasil_002 = ukur_spasial(0.02)
hasil_0001 = ukur_spasial(0.001)

print("Kotak 0,02 derajat")
print(hasil_002)
print("\nKotak 0,001 derajat")
print(hasil_0001)

Kotak 0,02 derajat
{'sisi_kotak': 0.02, 'waktu_numpy': 0.5019255120000707, 'waktu_strtree': 1.07400424799971, 'speedup': 0.4673403414696794, 'hasil_sama': True}

Kotak 0,001 derajat
{'sisi_kotak': 0.001, 'waktu_numpy': 0.49637950899978023, 'waktu_strtree': 0.013157046999822342, 'speedup': 37.727273377261845, 'hasil_sama': True}


### Pada NumPy, setiap kotak tetap harus membandingkan seluruh titik sehingga ukuran kotak tidak banyak mengurangi pekerjaan. Sebaliknya, STRtree dapat melewati kelompok titik yang kotaknya jelas berada di luar wilayah pencarian; semakin kecil kotak pencarian, semakin sedikit kandidat yang perlu diperiksa sehingga keuntungan indeks biasanya semakin besar. Modul menjelaskan prinsip ini sebagai penyaringan berdasarkan bounding box sebelum pemeriksaan lokasi secara tepat.

## 6. Jalankan sel di bawah setelah K-8. Jelaskan perbedaan time travel (membaca versi lama) dengan restore (mengembalikan tabel ke versi lama), dan mengapa restore dicatat sebagai versi baru, bukan menghapus riwayat.

In [16]:
from deltalake import DeltaTable

dt = DeltaTable(PATH_DELTA)
print(
    "versi sebelum restore:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)

# Mengembalikan kondisi tabel ke versi 1, tetapi dicatat sebagai versi BARU
dt.restore(1)
dt = DeltaTable(PATH_DELTA)
print(
    "versi setelah restore:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)
print(
    [h.get("operation") for h in dt.history()][:3]
)

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


### time travel digunakan untuk membaca kondisi tabel pada versi lama tanpa mengubah versi terbaru, sedangkan restore(1) digunakan untuk mengembalikan kondisi aktif tabel ke isi versi 1. Restore dicatat sebagai versi baru karena Delta Lake tidak menghapus riwayat transaksi sebelumnya; perubahan baru ditambahkan ke _delta_log/, sehingga riwayat versi tetap dapat ditelusuri. Contoh output pada modul adalah versi 23 dengan 160.000 baris sebelum restore, lalu versi 24 dengan 150.000 baris setelah restore, dengan operasi terbaru RESTORE.

# **Tugas Praktikum**

## 1. Rekap benchmark. Kumpulkan benchmark_nosql.csv dan tabel rekap K-9 yang sudah terisi, dengan satu kalimat interpretasi untuk tiap baris.

In [18]:
import os
import pandas as pd

PATH_BENCHMARK = f"{DIR_SIMPAN}/benchmark_nosql.csv"

assert os.path.exists(PATH_BENCHMARK), (
    "benchmark_nosql.csv belum ditemukan. "
    "Jalankan K-9 terlebih dahulu."
)
rekap = pd.read_csv(PATH_BENCHMARK)

def buat_interpretasi(langkah):
    s = langkah.lower()
    if "tanpa indeks" in s:
        return "Pencarian tanpa indeks memerlukan pemeriksaan lebih banyak baris sehingga waktunya paling tinggi."
    if "dengan indeks" in s:
        return "Indeks mempercepat pencarian berdasarkan id karena lokasi data dapat ditemukan secara langsung."
    if "key-value" in s and "cari" in s:
        return "Key-value cepat untuk pencarian langsung karena data diakses menggunakan key."
    if "key-value" in s and ("hitung" in s or "rata" in s):
        return "Key-value lebih lambat untuk agregasi karena setiap nilai harus dibuka dan diproses terlebih dahulu."
    if "duckdb" in s and ("hitung" in s or "sum" in s):
        return "DuckDB efisien untuk agregasi karena membaca kolom yang diperlukan dan memproses banyak baris secara vektor."
    if "sqlite" in s and ("hitung" in s or "sum" in s):
        return "SQLite memproses penyimpanan berbasis baris sehingga agregasi dapat memerlukan pembacaan data yang lebih besar."
    if "graph" in s:
        return "Graph memodelkan hubungan antar-zona secara langsung sehingga pertanyaan keterhubungan lebih sederhana."
    if "numpy" in s and "kotak" in s:
        return "NumPy memeriksa kondisi terhadap seluruh titik tanpa menggunakan indeks spasial."
    if "strtree" in s:
        return "STRtree mempercepat pencarian spasial dengan menyaring kandidat berdasarkan struktur indeks."
    if "compact" in s:
        return "Compaction menggabungkan banyak file kecil menjadi file yang lebih sedikit tanpa mengubah jumlah baris data."
    if "baca" in s:
        return "Waktu menunjukkan biaya pembacaan data pada kondisi dan lingkungan pengujian saat praktikum."
    return "Waktu menunjukkan biaya proses pada kondisi dan lingkungan pengujian saat praktikum."

rekap["interpretasi"] = rekap["langkah"].apply(buat_interpretasi)

print("=== REKAP BENCHMARK K-9 ===")
print(rekap.to_string(index=False))

PATH_REKAP = f"{DIR_SIMPAN}/rekap_K9_lengkap.csv"
rekap.to_csv(PATH_REKAP, index=False)
print(f"\nRekap lengkap disimpan di: {PATH_REKAP}")

=== REKAP BENCHMARK K-9 ===
                                            langkah   detik                                                                                                    interpretasi
                                  baca trips_bersih  3.5111                    Waktu menunjukkan biaya pembacaan data pada kondisi dan lingkungan pengujian saat praktikum.
                             relasional: muat tabel  1.6004                            Waktu menunjukkan biaya proses pada kondisi dan lingkungan pengujian saat praktikum.
                             key-value: bangun dict  3.6402                            Waktu menunjukkan biaya proses pada kondisi dan lingkungan pengujian saat praktikum.
             cari 200 perjalanan - SQL tanpa indeks  4.9507               Pencarian tanpa indeks memerlukan pemeriksaan lebih banyak baris sehingga waktunya paling tinggi.
            cari 200 perjalanan - SQL dengan indeks  0.0053                 Indeks mempercepat pencarian berdasa

### Kode membaca benchmark_nosql.csv hasil K-9, kemudian menambahkan kolom interpretasi pada setiap baris sehingga setiap hasil pengukuran memiliki satu kalimat penjelasan. Angka yang muncul harus menggunakan hasil benchmark milik sendiri karena modul menegaskan bahwa waktu dapat berbeda pada setiap komputer dan dataset.

## 2. Data lain, cara lain. Pilih satu data lain (mis. “zona”, “hari”, atau “metode bayar”) dan rancang tiga cara menyimpannya (tabel, document, dan graph atau key-value). Untuk masing-masing, tulis satu pertanyaan yang mudah dan satu yang canggung dijawab.

In [19]:
import json
import sqlite3
from tinydb import TinyDB
from tinydb.storages import MemoryStorage

data_pembayaran = sample[
    ["metode_bayar", "total_bayar", "wilayah_jemput"]
].copy()

ringkasan = (
    data_pembayaran
    .groupby("metode_bayar")
    .agg(
        jumlah_transaksi=("metode_bayar", "size"),
        rata_rata_bayar=("total_bayar", "mean"),
        total_pendapatan=("total_bayar", "sum")
    )
    .reset_index()
)
con_tugas = sqlite3.connect(":memory:")

ringkasan.to_sql(
    "metode_pembayaran",
    con_tugas,
    index=False,
    if_exists="replace"
)

hasil_tabel = con_tugas.execute("""
    SELECT metode_bayar, jumlah_transaksi
    FROM metode_pembayaran
    WHERE metode_bayar = 'Tunai'
""").fetchone()

hasil_tabel_canggung = con_tugas.execute("""
    SELECT metode_bayar, total_pendapatan
    FROM metode_pembayaran
    ORDER BY total_pendapatan DESC
    LIMIT 1
""").fetchone()

db_doc = TinyDB(storage=MemoryStorage)
tabel_doc = db_doc.table("metode_bayar")
docs = [
    {
        "metode_bayar": row.metode_bayar,
        "jumlah_transaksi": int(row.jumlah_transaksi),
        "rata_rata_bayar": float(row.rata_rata_bayar),
        "total_pendapatan": float(row.total_pendapatan)
    }
    for row in ringkasan.itertuples()
]
tabel_doc.insert_multiple(docs)

doc_tunai = tabel_doc.get(
    lambda d: d["metode_bayar"] == "Tunai"
)

doc_terbesar = max(
    tabel_doc.all(),
    key=lambda d: d["total_pendapatan"]
)

kv_metode = {
    f"metode:{row.metode_bayar}": json.dumps({
        "metode_bayar": row.metode_bayar,
        "jumlah_transaksi": int(row.jumlah_transaksi),
        "rata_rata_bayar": float(row.rata_rata_bayar),
        "total_pendapatan": float(row.total_pendapatan)
    })
    for row in ringkasan.itertuples()
}
kv_tunai = json.loads(kv_metode["metode:Tunai"])

semua_kv = [
    json.loads(value)
    for value in kv_metode.values()
]

kv_terbesar = max(
    semua_kv,
    key=lambda d: d["total_pendapatan"]
)
print("=== DATA METODE PEMBAYARAN ===")
print(ringkasan)
print("\n--- TABEL ---")
print("Pertanyaan mudah :", hasil_tabel)
print("Pertanyaan canggung :", hasil_tabel_canggung)
print("\n--- DOCUMENT ---")
print("Pertanyaan mudah :", doc_tunai)
print("Pertanyaan canggung :", doc_terbesar)
print("\n--- KEY-VALUE ---")
print("Pertanyaan mudah :", kv_tunai)
print("Pertanyaan canggung :", kv_terbesar)

=== DATA METODE PEMBAYARAN ===
   metode_bayar  jumlah_transaksi  rata_rata_bayar  total_pendapatan
0        Gratis               943        22.913075          21607.03
1  Kartu kredit            239237        28.118488        6726982.62
2      Sengketa              1545        24.609120          38021.09
3         Tunai             51898        23.476644        1218390.85

--- TABEL ---
Pertanyaan mudah : ('Tunai', 51898)
Pertanyaan canggung : ('Kartu kredit', 6726982.62)

--- DOCUMENT ---
Pertanyaan mudah : {'metode_bayar': 'Tunai', 'jumlah_transaksi': 51898, 'rata_rata_bayar': 23.476643608616904, 'total_pendapatan': 1218390.85}
Pertanyaan canggung : {'metode_bayar': 'Kartu kredit', 'jumlah_transaksi': 239237, 'rata_rata_bayar': 28.118487608522095, 'total_pendapatan': 6726982.62}

--- KEY-VALUE ---
Pertanyaan mudah : {'metode_bayar': 'Tunai', 'jumlah_transaksi': 51898, 'rata_rata_bayar': 23.476643608616904, 'total_pendapatan': 1218390.85}
Pertanyaan canggung : {'metode_bayar': 'Kartu

### Tabel cocok untuk pertanyaan yang membutuhkan penyaringan dan agregasi karena data dapat langsung diproses dengan SQL. Document cocok ketika satu objek ingin disimpan dalam bentuk yang fleksibel, sedangkan key-value sangat sederhana untuk mengambil satu data berdasarkan key tetapi menjadi canggung ketika harus membandingkan atau menghitung seluruh nilai karena semua value perlu dibuka terlebih dahulu. Hal ini sejalan dengan penjelasan modul bahwa key-value unggul pada pencarian berdasarkan key, tetapi kurang cocok untuk pencarian atau perhitungan berdasarkan isi data.

## 3. Delta Lake dua hari. Simulasikan dua hari kerja: hari pertama menulis sebagian data, hari kedua menambah data dan mengoreksi beberapa baris. Tunjukkan bahwa Anda bisa membaca tabel seperti keadaannya di akhir hari pertama, dan lampirkan isi _delta_log/.

In [27]:
import os
import shutil
import json
from pathlib import Path
from deltalake import DeltaTable, write_deltalake

# Kolom yang digunakan
KOLOM_DELTA = [
    "waktu_jemput",
    "zona_jemput",
    "zona_tujuan",
    "wilayah_jemput",
    "jarak_mil",
    "total_bayar",
    "metode_bayar"
]

# Siapkan data
data_tugas_delta = (
    df[KOLOM_DELTA]
    .sample(
        n=min(200_000, len(df)),
        random_state=42
    )
    .reset_index(drop=True)
)

# PATH LOCAL Delta Lake dikerjakan di storage lokal
PATH_DELTA_LOCAL = "/content/delta_dua_hari"

# Hapus hasil lama bila ada
if os.path.exists(PATH_DELTA_LOCAL):
    shutil.rmtree(PATH_DELTA_LOCAL)

# HARI 1 Menulis 100.000 baris
hari1 = data_tugas_delta.iloc[:100_000].copy()
write_deltalake(
    PATH_DELTA_LOCAL,
    hari1,
    mode="overwrite"
)
dt = DeltaTable(PATH_DELTA_LOCAL)

versi_hari1 = dt.version()
jumlah_hari1 = len(dt.to_pandas())
print("========== HARI 1 ==========")
print("Versi :", versi_hari1)
print("Baris :", jumlah_hari1)

# HARI 2 Menambah 50.000 baris
hari2 = data_tugas_delta.iloc[100_000:150_000].copy()
write_deltalake(
    PATH_DELTA_LOCAL,
    hari2,
    mode="append"
)
dt = DeltaTable(PATH_DELTA_LOCAL)

print("\n========== HARI 2 ==========")
print("Versi :", dt.version())
print("Baris :", len(dt.to_pandas()))

# HARI 2 - KOREKSI DATA Contoh: tarif Tunai di Bronx + 1.0
sekarang = data_tugas_delta.iloc[:150_000]
n_koreksi = int(
    (
        (sekarang["wilayah_jemput"] == "Bronx") &
        (sekarang["metode_bayar"] == "Tunai")
    ).sum()
)
print("\n========== KOREKSI ==========")
print("Jumlah baris yang akan dikoreksi :", n_koreksi)

dt.update(
    predicate=(
        "wilayah_jemput = 'Bronx' "
        "AND metode_bayar = 'Tunai'"
    ),
    updates={
        "total_bayar": "total_bayar + 1.0"
    }
)
dt = DeltaTable(PATH_DELTA_LOCAL)
print("Versi setelah koreksi :", dt.version())
print("Jumlah baris terbaru   :", len(dt.to_pandas()))

# TIME TRAVEL Membaca kondisi pada akhir Hari 1
v_hari1 = DeltaTable(
    PATH_DELTA_LOCAL,
    version=versi_hari1
).to_pandas()

v_terbaru = DeltaTable(
    PATH_DELTA_LOCAL
).to_pandas()
print("\n========== TIME TRAVEL ==========")
print("Baris versi Hari 1 :", len(v_hari1))
print("Baris versi terbaru :", len(v_terbaru))

# HISTORY
print("\n========== HISTORY ==========")
dt = DeltaTable(PATH_DELTA_LOCAL)
for h in dt.history():
    print(
        "versi:",
        h.get("version"),
        "| operasi:",
        h.get("operation")
    )

# ISI _delta_log
LOG_DIR = Path(PATH_DELTA_LOCAL) / "_delta_log"
print("\n========== _delta_log ==========")
for file_log in sorted(LOG_DIR.glob("*.json")):
    print("\n---", file_log.name, "---")
    with open(
        file_log,
        "r",
        encoding="utf-8"
    ) as f:
        for nomor_baris, baris in enumerate(f, start=1):
            baris = baris.strip()
            if not baris:
                continue
            try:
                isi = json.loads(baris)
                print(
                    f"\nBaris {nomor_baris}:"
                )
                print(
                    json.dumps(
                        isi,
                        indent=2
                    )
                )
            except json.JSONDecodeError:
                print(
                    f"Baris {nomor_baris} "
                    "bukan JSON yang valid."
                )

# SALIN HASIL DELTA KE GOOGLE DRIVE
PATH_DRIVE_DELTA = f"{DIR_SIMPAN}/delta_dua_hari"

# Hapus hasil sebelumnya bila ada
if os.path.exists(PATH_DRIVE_DELTA):
    shutil.rmtree(PATH_DRIVE_DELTA)

shutil.copytree(
    PATH_DELTA_LOCAL,
    PATH_DRIVE_DELTA
)
print(
    "\nFolder Delta berhasil disalin ke:"
)
print(PATH_DRIVE_DELTA)

# SIMPAN ISI _delta_log SEBAGAI TXT
PATH_LOG_TXT = f"{DIR_SIMPAN}/delta_log_lampiran.txt"
with open(
    PATH_LOG_TXT,
    "w",
    encoding="utf-8"
) as out:
    for file_log in sorted(LOG_DIR.glob("*.json")):
        out.write(
            f"\n========== {file_log.name} ==========\n"
        )
        with open(
            file_log,
            "r",
            encoding="utf-8"
        ) as f:
            out.write(f.read())
print(
    "\nIsi _delta_log disimpan di:"
)
print(PATH_LOG_TXT)

# BUAT ZIP UNTUK PENGUMPULAN
PATH_ZIP = f"{DIR_SIMPAN}/delta_dua_hari"
shutil.make_archive(
    PATH_ZIP,
    "zip",
    PATH_DRIVE_DELTA
)
print(
    "\nArsip ZIP berhasil dibuat:"
)
print(f"{PATH_ZIP}.zip")

========== HARI 1 ==========
Versi : 0
Baris : 100000

========== HARI 2 ==========
Versi : 1
Baris : 150000

========== KOREKSI ==========
Jumlah baris yang akan dikoreksi : 20
Versi setelah koreksi : 2
Jumlah baris terbaru   : 150000

========== TIME TRAVEL ==========
Baris versi Hari 1 : 100000
Baris versi terbaru : 150000

========== HISTORY ==========
versi: 2 | operasi: UPDATE
versi: 1 | operasi: WRITE
versi: 0 | operasi: WRITE

========== _delta_log ==========

--- 00000000000000000000.json ---

Baris 1:
{
  "commitInfo": {
    "timestamp": 1790841532339,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "Overwrite"
    },
    "engineInfo": "delta-rs:py-1.6.6",
    "operationMetrics": {
      "num_added_files": 1,
      "num_removed_files": 0,
      "num_partitions": 0,
      "num_added_rows": 100000,
      "execution_time_ms": 106,
      "num_retries": 0
    },
    "clientVersion": "delta-rs.py-1.6.6"
  }
}

Baris 2:
{
  "protocol": {
    "minReaderVersion": 

### pada simulasi ini, versi pertama berisi data Hari 1, versi berikutnya bertambah setelah data Hari 2 ditulis, kemudian koreksi menghasilkan versi baru lagi tanpa menghapus versi sebelumnya. Dengan DeltaTable(..., version=versi_hari1), kondisi tabel pada akhir Hari 1 tetap dapat dibaca melalui time travel. Folder _delta_log/ menyimpan catatan transaksi tiap versi, sehingga perubahan dapat dilacak dan data lama tetap tersedia. Modul juga menunjukkan pola yang sama pada K-7: versi 0 berisi 100.000 baris, versi 1 menjadi 150.000 baris, dan update menghasilkan versi 2.

## Laporan rancangan penyimpanan untuk studi kasus di Bagian P (PDF), dan Restart notebook and run all pada runtime bersih.

## Berada di laporan